# 大模型微调（Fine-Tuning）与 PEFT（高效参数微调）
在掌握了预训练、Prompt 工程、RAG、Agent 编排与本地部署之后，微调 是将通用的开源大模型改造为特定领域（如医疗、法律、金融或专属 Agent 格式输出）“超级专家”的关键钥匙。

## 微调分类与 LoRA 低秩分解数学原理

1. 对比全量微调 vs. 高效参数微调（PEFT）：理解显存开销、梯度更新范围与适用场景的差异。
2. 深入拆解 LoRA（Low-Rank Adaptation）底层数学原理：理解低秩矩阵分解 $W = W_0 + \Delta W = W_0 + B \cdot A$ 的前向传播与反向传播。
3. 掌握 LoRA 核心超参数（$r$, $\alpha$, dropout, target_modules）：搞懂在真实工程中如何进行超参数选型与显存预算评估。

#### LoRA (Low-Rank Adaptation) 底层数学原理
在传统的全量微调（Full Fine-Tuning）中，需要对模型的所有参数矩阵（如 7B 模型对应的 70 亿参数）更新梯度，这不仅需要极其庞大的 GPU 显存（保存优化器状态 AdamW state、梯度 Gradients 等），还极易导致模型产生灾难性遗忘（Catastrophic Forgetting）。

1. 低秩矩阵分解的数学公式
    * 假设原始 Transformer 层的权重矩阵为 $W_0 \in \mathbb{R}^{d \times k}$。LoRA 不直接修改 $W_0$，而是将其冻结（Freeze），并通过两个低秩矩阵 $A \in \mathbb{R}^{r \times k}$ 和 $B \in \mathbb{R}^{d \times r}$ 来近似表示参数更新量 $\Delta W$：
      $$\Delta W = B \cdot A \quad (\text{其中秩 } r \ll \min(d, k))$$
    * 在前向传播时，输入向量 $x$ 的计算方式变为：
      $$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} (B A) x$$
            * 初始化技巧：$A$ 矩阵采用高斯分布初始化，$B$ 矩阵初始化为全零（Zero Initialization）。这样在训练刚开始时 $\Delta W = 0$，保证微调初始状态与原模型完全等价。
            * 缩放因子 $\frac{\alpha}{r}$：通过常数 $\alpha$ 对 LoRA 更新量进行缩放，使在改变秩 $r$ 时无需大幅重新调整学习率。

2. 显存开销对比（以 7B 模型为例）

   | 维度          | 全量微调 (Full FT)                                     | LoRA 微调 (PEFT)                                            |
   |-------------|----------------------------------------------------|-----------------------------------------------------------|
   | 可训练参数量      | 100% (7,000,000,000)                               | 0.05% ~ 0.2% (~10,000,000)                                |
   | 显存需求 (BF16) | $\approx 80\text{GB} \sim 110\text{GB}$ (需多卡 A100) | $\approx 14\text{GB} \sim 18\text{GB}$ (单张 RTX 3090/4090) |
   | 部署与权重融合     | 保存全量权重 (14GB)                                      | 仅保存 LoRA 适配器 (仅数十 MB)，可动态无缝融回原权重                          |


#### 使用 Hugging Face peft 库为 Qwen2.5 挂载 LoRA 适配器
> 环境准备提示：运行本段代码需要安装 transformers 与 peft 库：
`pip install torch transformers peft accelerate`

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

# ================= 1. 指定基座模型 =================
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print("🚀 正在加载 Base Model ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# ================= 2. 定义 LoRA 配置 (LoraConfig) =================
lora_config = LoraConfig(
    r=16,                         # 低秩矩阵的秩 (Rank)，常用 8, 16, 32, 64
    lora_alpha=32,                # 缩放系数 alpha，通常设为 2 * r
    target_modules=[              # 指定挂载 LoRA 的 Transformer 模块
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,            # 防止过拟合的 Dropout 比例
    bias="none",                  # 不训练偏置项
    task_type=TaskType.CAUSAL_LM  # 任务类型：因果语言模型
)

# ================= 3. 注入 LoRA 适配器并打印可训练参数比例 =================
peft_model = get_peft_model(model, lora_config)

print("\n📊 LoRA 参数统计:")
peft_model.print_trainable_parameters()

# 示例：查看模型结构中被替换为 lora.Linear 的层
print("\n🔍 检查第一个 Self-Attention 层的 Q 投影阵:")
print(peft_model.base_model.model.model.layers[0].self_attn.q_proj)


**核心工程超参数选型指南**
1. `r` (Rank 秩)：
    * 简单领域分类/格式对齐：`r = 8` 或 `r = 16`。
    * 复杂领域知识注入（如医疗诊断、代码生成）：`r = 32` 或 `r = 64`。

2. `target_modules` 选择：
    * 早期 LoRA 仅作用于 Attention 的 $Q, V$ 矩阵；但在现代 LLM 微调中，同时作用于 Attention ($q, k, v, o$) 和 MLP 层 ($gate, up, down\_proj$) 能够带来显著更好的微调效果。

3. 零额外推理延迟 (Zero-latency Inference)：
    * 在微调结束后，可以通过 `peft_model.merge_and_unload()` 将 LoRA 的 $B \cdot A$ 权重物理加回到原始权重 $W_0$ 中，从而生成一个合并后的新权重，在推理阶段不增加任何额外的计算延迟！

**思考**
1. QLoRA 原理预习：如果显存极其有限（例如只有 8GB/12GB 显存），如何在 LoRA 的基础上结合 4-bit NF4 (NormalFloat4) 量化，衍生出 QLoRA 架构？
2. 权重融合与解耦：假设你为客服系统微调了一个“法律场景 LoRA”和一个“IT 运维 LoRA”，在生产部署时，能否在同一个基座模型（Base Model）上动态切换这两个 LoRA 适配器？这样做相比部署两个独立的大模型能节省多少显存？

## QLoRA（Quantized Low-Rank Adaptation）与 4-bit 量化微调
QLoRA 的三大核心突破，并实战使用 `bitsandbytes` 与 `peft` 在单卡上加载 4-bit 量化基座模型并挂载 LoRA 进行训练。

#### QLoRA 三大核心黑科技

1. NF4（NormalFloat4）信息论最优数据类型
    * 传统 INT4 或 FP4 量化假定数据分布是均匀的，但预训练大模型的权重通常呈现正态分布（Normal Distribution）。
    * NF4 是一种针对正态分布数据在信息论上最优的分位点量化数据类型（Quantile Quantization）。它确保 4-bit 的 16 个离散量化桶（Quantization Bins）中包含等概率的参数点，从而极大降低了 4-bit 量化带来的信息熵损失。

2. 双重量化（Double Quantization, DQ）
    * 量化本身需要记录量化缩放因子（Quantization Constants/Scales）。在传统量化中，这些缩放因子依然占用较多显存。
    * 双重量化对量化缩放因子本身再次进行二次 8-bit 量化，将平均每个权重的量化元数据开销从 0.5 bit/param 压缩到了惊人的 0.127 bit/param，为 7B/13B 模型节省了近 0.5GB~1GB 的额外显存。

3. 分页优化器（Paged Optimizers）
    * 借鉴了操作系统（OS）内存分页与 vLLM 显存页的思想，利用 NVIDIA Unified Memory 特性。
    * 当微调过程中长文本训练 Batch 触发显存峰值（VRAM Spike）时，将优化器状态（Optimizer States）无缝页出（Page-out）到系统 CPU 内存（RAM）中；当计算需要时再页入（Page-in）到 GPU 显存，有效防止训练过程突然 OOM 崩溃。

#### 使用 `bitsandbytes `+ `peft` 配置 QLoRA 4-bit 训练模型
> 环境准备提示：运行本段代码需要安装 PyTorch、transformers、peft 和 bitsandbytes 库：
`pip install torch transformers peft bitsandbytes accelerate`

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# ================= 1. 配置 4-bit NF4 量化参数 (BitsAndBytesConfig) =================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                        # 启用 4-bit 量化加载
    bnb_4bit_quant_type="nf4",                # 指定使用信息论最优的 NF4 数据类型
    bnb_4bit_compute_dtype=torch.bfloat16,   # 计算时将 4-bit 权重反量化为 BF16 进行矩阵乘法
    bnb_4bit_use_double_quant=True            # 开启双重量化 (Double Quantization)
)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print("🚀 正在加载 4-bit NF4 压缩基座模型...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# 加载量化基座模型 (显存仅需约 5GB)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# ================= 2. 预处理量化模型以支持梯度检查点 (Layer Norm 冻结等) =================
model = prepare_model_for_kbit_training(model)

# ================= 3. 配置 QLoRA 的 LoraConfig =================
qlora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# 挂载 QLoRA 适配器
peft_model = get_peft_model(model, qlora_config)

print("\n📊 QLoRA 训练参数与显存开销统计:")
peft_model.print_trainable_parameters()

# 查看 GPU 显存占用情况
if torch.cuda.is_available():
    vram_allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    print(f"💡 当前基座模型占用显存: {vram_allocated:.2f} GB (仅为全量 FP16 的 1/3 以下!)")


**核心工程细节解析**
1. `prepare_model_for_kbit_training()` 的必要性：
    * 在对 4-bit 量化模型挂载 LoRA 进行训练前，必须调用此函数。它会将 LayerNorm 层转型为 FP32 以保证数值稳定性，并开启冻结权重的 `requires_grad=False` 与梯度检查点（Gradient Checkpointing）兼容支持。

2. 计算数据类型与存储数据类型的分离：
    * `load_in_4bit=True` 确保权重在显存中以 4-bit NF4 格式存储（极大省显存）；而 `bnb_4bit_compute_dtype=torch.bfloat16` 确保在 Tensor Core 进行前向与反向传播乘法时，动态反量化为 BF16 进行计算（保证训练精度不崩溃）。

QLoRA 的三大核心创新（NF4、双重量化、分页优化器），并成功实现了在极低显存下加载 4-bit 模型并注入可训练 LoRA 矩阵的工程配置。
**思考：**
1. 显存峰值与 Batch Size：在 QLoRA 训练中，虽然模型权重被压缩到了 4-bit，但训练过程中的 Activation Memory（激活值显存） 会随着 `batch_size` 和 `max_seq_length` 的增加而急剧上升。如果在单卡 16GB 显存上训练发生 OOM，有哪些手段（如 Gradient Accumulation 梯度累加、Gradient Checkpointing 梯度检查点）可以降低激活值显存占用？
2. QLoRA 保存与导出：训练完成后，`peft_model.save_pretrained("./qlora_output")` 保存的文件包含基座模型的 4-bit 权重吗？如果想要将 QLoRA 导出为 GGUF 格式供 Ollama 使用，正确的合并与导出步骤是什么？

## SFT（Supervised Fine-Tuning，指令监督微调）
SFT（Supervised Fine-Tuning，指令监督微调）数据集构建与数据预处理流水线（Pipeline）是决定大模型微调成败的最核心因素。

#### SFT 数据集构建规范与对话模板 Mask 机制
1. 掌握 SFT 数据集 Schema 设计：理清单轮（Single-turn）与多轮（Multi-turn）对话数据集的标准 JSON / JSONL 结构。
2. 深入理解 Prompt / System Masking 机制：搞懂为什么在 SFT 计算 Loss 时，必须 Mask 掉 System 和 User 的 Prompt，只对 Assistant 的 Response 计算交叉熵损失（Cross-Entropy Loss）。
3. 使用 PyTorch / Transformers 编写高效率的 SFT 数据 Tokenizer 预处理 Pipeline。

#### SFT 损失计算与 Response-Only Masking 机制

**1.为什么不能对全量 Token 计算 Loss？**

在预训练（Pre-training）阶段，模型的目标是预测下一个 Token（Causal Language Modeling），因此会对输入的所有 Token 计算 Loss。
但在 SFT（指令监督微调） 阶段，我们的目标是让模型学会“根据用户的指令/上下文回答问题”。

* 如果对 Prompt（System + User 输入）也计算 Loss：模型会花费大量的梯度更新去“背诵/记忆”用户的提问习惯或固定 Prompt，导致模型生成能力下降，甚至学会自我幻觉。
* 正确的做法（Response-Only Loss）：在前向传播拼接序列 `[System, User, Assistant]` 时，将 `System` 和 `User` 部分的 Target Label 全部置为 `-100`（PyTorch `CrossEntropyLoss` 的默认 `ignore_index`），仅保留 `Assistant` 回答部分的真实 Token ID 来计算 Loss。


        输入序列 (Input IDs):   [<|im_start|>, user, \n, 帮我写代码, <|im_end|>, \n, <|im_start|>, assistant, \n, print('hello'), <|im_end|>]
        目标标签 (Labels):      [   -100,      -100, -100,    -100,      -100,    -100,     -100,        -100,     \n,    print,   ('hello'),   <|im_end|>]
        └─── 被 Mask 掉 (ignore_index=-100，不计 Loss) ───┘ └─── 仅对 Assistant 生成部分计算 Loss ───┘




**2. 标准 SFT 数据集格式（JSONL）**

在工程落地的 Agent/领域微调中，推荐统一采用 OpenAI / Hugging Face 规范的 `messages` 数组格式：

```
{"messages": [{"role": "system", "content": "你是一个严谨的 Python 代码审查助手。"}, {"role": "user", "content": "这行代码有什么问题？print x"}, {"role": "assistant", "content": "在 Python 3 中，print 是一个函数而非关键字，必须使用括号：`print(x)`。"}]}
```

#### 编写带 Response-Only Mask 的 SFT 数据处理流水线
> 环境准备提示：运行本段代码需要安装 PyTorch 与 transformers 库：
`pip install torch transformers`

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

# ================= 1. 模拟领域 SFT 原始数据集 =================
raw_dataset = [
    {
        "messages": [
            {"role": "system", "content": "你是一个金融量化交易专家。"},
            {"role": "user", "content": "什么是夏普比率（Sharpe Ratio）？"},
            {"role": "assistant", "content": "夏普比率是衡量基金或投资组合‘每承受一单位总风险所产生的超额回报’的指标。计算公式为：(投资组合预期收益率 - 无风险利率) / 投资组合标准差。"}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "你是一个 Agent 架构调优专家。"},
            {"role": "user", "content": "ReAct 模式的核心三个步骤是什么？"},
            {"role": "assistant", "content": "ReAct 的核心循环步骤为：1. Thought（思考）；2. Action（行动，如调用 Tool）；3. Observation（观察工具返回结果）。"}
        ]
    }
]

# ================= 2. 编写带 Response-Only Label Mask 的 PyTorch Dataset =================
class SFTDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        messages = self.data[index]["messages"]

        # 1. 提取出 Prompt 部分（System + User）与全量对话
        prompt_messages = [m for m in messages if m["role"] != "assistant"]

        # 2. 使用 Tokenizer 的 apply_chat_template 构造带有特定 Special Token 的文本
        full_text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        prompt_text = self.tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)

        # 3. 分别编码全量文本与 Prompt 文本
        full_encoded = self.tokenizer(full_text, truncation=True, max_length=self.max_length, return_tensors="pt")
        prompt_encoded = self.tokenizer(prompt_text, truncation=True, max_length=self.max_length, return_tensors="pt")

        input_ids = full_encoded["input_ids"].squeeze(0)
        attention_mask = full_encoded["attention_mask"].squeeze(0)

        # 4. 构建 Labels，默认全部设为 -100 (Ignore)
        labels = input_ids.clone()
        prompt_len = prompt_encoded["input_ids"].shape[1]

        # 将 Prompt 对应的位置全部置为 -100，只保留 Assistant 响应部分计算梯度
        labels[:prompt_len] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

# ================= 3. 验证 Masking 机制输出 =================
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

sft_dataset = SFTDataset(raw_dataset, tokenizer)
sample = sft_dataset[0]

print("🔍 验证数据 Token 化与 Label Masking 结果:")
print(f"Input IDs 长度: {len(sample['input_ids'])}")
print(f"Labels 数组（前 15 个 Token 的 Label）: {sample['labels'][:15].tolist()}")
print(f"💡 包含 -100 的 Token 数量 (即被 Mask 掉的 Prompt 部分): {(sample['labels'] == -100).sum().item()}")


**核心工程细节解析**

1. `add_generation_prompt` 的技巧：

    在拼接 Prompt 文本时，设置 `add_generation_prompt=True` 会自动在 User 消息末尾补齐当前模型的 Assistant 引导头（例如 `<|im_start|>assistant\n）`，这样在计算 Prompt 长度时能精准切分到模型即将吐出第一个回复 Token 的物理位置

2. 多轮对话（Multi-Turn）的 Mask 处理：
    对于包含多轮交互的复杂数据（User1 $\rightarrow$ Assistant1 $\rightarrow$ User2 $\rightarrow$ Assistant2），预处理代码逻辑需要通过遍历 Token 列表，精准定位每一次 `User` 说话的起点与终点，将其分段全部 Mask 为 `-100`，只开放每一轮 `Assistant` 的 Token 进行 Loss 求解。

**思考**

1. 数据清洗与过滤：在构建 Agent Tool Calling（工具调用）的 SFT 数据集时，如果训练集中包含很多模型格式错误（如 JSON 语法错误、缺少引号）的样本，会对微调后的模型造成什么后果？应该在 Data Pipeline 中配置哪些正则校验或 JSON Schema 校验算子？
2. 数据量与过拟合：对于特定的领域 Agent 微调（例如“专写 SQL 语句的 Agent”），高性能的 SFT 通常需要多少条高质量的数据？（提示：质量远比数量重要，1000~3000 条经过精打细细磨的高质量 SFT 数据往往优于 5 万条低质数据）。